# Reclamações 02 · Camada Silver — Manifestações

## 1. Objetivo e método

Este notebook transforma a Bronze de manifestações nos dois objetos que a Gold consome: `fato_manifestacao`, com as quantidades por distribuidora, tipologia, nível e mês, e `controle_anomalias_manifestacao`, que registra o teste de meses anômalos e o que foi imputado. As três dimensões do modelo já existem: `dim_distribuidora` (notebook `base/04_silver_dimensoes`), `dim_tipologia` e `dim_tempo` (notebook `complaints/01_silver_reference`).

### O que a Silver resolve

| Achado na Bronze | Tratamento nesta etapa |
|---|---|
| Série de 2023 com codificação anterior à REH 2.992/2021 e mês de julho de 2026 ainda não consolidado | Recorte de janeiro de 2024 a junho de 2026, pela junção com `dim_tempo` |
| Linhas repetidas com todos os campos idênticos | Remoção das cópias exatas, mesma regra aplicada à continuidade e ao INDGER |
| Todos os campos publicados como texto | CNPJ com 14 caracteres, código de tipologia como texto, nível e quantidades como número |
| Códigos que não existem na REH | Excluídos por junção interna com `dim_tipologia` e listados na seção 5 |
| Linhas de subtotal e famílias que não são reclamação | Excluídas pelo atributo `bloco` da `dim_tipologia` |
| Município no grão, sem uso nas perguntas de negócio | Agregação para distribuidora × tipologia × nível × mês |
| Meses com volume fora do padrão da própria distribuidora | Detecção por bloco e nível, sem marcar salto que persiste nos meses seguintes; imputação da mediana local no comercial estrito do nível 1, com o valor publicado preservado ao lado |
| Distribuidora com série internamente inconsistente | Registro em `exclusao_ranking_manifestacao`, com o motivo e a evidência, para exclusão na Gold |

### O que a Silver não faz

Não filtra o universo de distribuidoras nem calcula indicador. O recorte de grande porte, a exclusão de distribuidoras e o cálculo por janela móvel pertencem à Gold; a Silver apenas registra a exclusão e a evidência que a sustenta. A única exceção é a detecção de meses anômalos, restrita às distribuidoras de grande porte: os limites foram calibrados nesse universo, e nas empresas pequenas a variação natural de volumes baixos marcaria meses sem anomalia real.

### Tabelas produzidas

| Tabela | Grão | Papel |
|---|---|---|
| `controle_anomalias_manifestacao` | CNPJ × nível × bloco × ano-mês, só grande porte | Totais do bloco, medianas de referência, resultado de cada teste e marcação de imputação |
| `fato_manifestacao` | CNPJ × tipologia × nível × ano-mês | Quantidades usadas (`qtd_*`), quantidades publicadas (`qtd_*_publicada`) e flag `imputado` |
| `exclusao_ranking_manifestacao` | CNPJ | Distribuidoras excluídas do ranking de reclamações, com motivo e evidência |

### Ordem de execução

Este notebook roda depois de `base/02_bronze_ingestion`, `base/04_silver_dimensoes` e `complaints/01_silver_reference`, e antes de `complaints/03_gold_ranking`.

## 2. Configuração

In [ ]:
import os
import sys

from pyspark.sql import functions as F
from pyspark.sql import Window

# Walk up from the working directory until the folder holding `src` is found,
# so the notebook works at any depth inside notebooks/
REPO_ROOT = os.getcwd()
while not os.path.isdir(os.path.join(REPO_ROOT, "src")):
    parent = os.path.dirname(REPO_ROOT)
    if parent == REPO_ROOT:
        raise FileNotFoundError("Repository root with a src folder not found above " + os.getcwd())
    REPO_ROOT = parent
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.config import CATALOG, INGESTION_COLUMNS, SCHEMA_BRONZE, SCHEMA_SILVER

BRONZE = f"{CATALOG}.{SCHEMA_BRONZE}"
SILVER = f"{CATALOG}.{SCHEMA_SILVER}"
TABELA_BRONZE = f"{BRONZE}.complaints"

# Channel label published by ANEEL -> service level (1 = call centre, 2 = ombudsman)
NIVEL_POR_CANAL = {"Nível 1": 1, "Nível 2": 2}

# Blocks of dim_tipologia that hold detailed complaints (family 102, no subtotal)
BLOCOS_RECLAMACAO = ["comercial_estrito", "rede_e_qualidade_outros", "tecnico"]

MEDIDAS = ["recebidas", "procedentes", "improcedentes"]

# Anomaly test: local median over 3 months before and 3 after, and median of the
# whole series of the same company, level and block
MESES_VIZINHOS = 3
LIMITE_INFERIOR_LOCAL = 0.40
LIMITE_SUPERIOR_LOCAL = 2.50
LIMITE_INFERIOR_SERIE = 0.20

# Only level 1 of the strict commercial block is imputed; the other blocks and level 2
# carry the flag in the control table and keep the published values
NIVEL_IMPUTADO = 1
BLOCOS_IMPUTADOS = ["comercial_estrito"]

# Companies excluded from the complaints ranking; the evidence is computed from the
# data in section 11 and stored next to the reason
EXCLUSOES_RANKING = {
    "08336783000190": "Serie do comercial estrito internamente inconsistente: procedentes acima "
                      "de recebidas em varios meses e volumes que a imputacao nao corrige",   # CELESC
}

# Grain of the fact table and of the anomaly control table
CHAVE_FATO = ["num_cnpj", "cod_tipologia", "nivel", "ano_mes"]
CHAVE_BLOCO = ["num_cnpj", "nivel", "bloco", "ano_mes"]

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA_SILVER}")

print(f"Origem..........: {TABELA_BRONZE}")
print(f"Destino.........: {SILVER}")
print(f"Teste local.....: fora de {LIMITE_INFERIOR_LOCAL:.0%} a {LIMITE_SUPERIOR_LOCAL:.0%} "
      f"da mediana de {MESES_VIZINHOS} meses antes e {MESES_VIZINHOS} depois")
print(f"Teste da serie..: abaixo de {LIMITE_INFERIOR_SERIE:.0%} da mediana da serie inteira")
print(f"Imputacao.......: nivel {NIVEL_IMPUTADO}, blocos {BLOCOS_IMPUTADOS}")

In [ ]:
def mediana(lista):
    """Exact median of a numeric array column; null when the array is empty.

    Spark has no exact median over a sliding window that skips the current row, so the
    neighbour values are collected into an array and the median is taken from it.
    """
    ordenada = F.array_sort(lista)
    n = F.size(ordenada)
    meio = (n / 2).cast("int")
    return (F.when(n == 0, F.lit(None).cast("double"))
             .when(n % 2 == 1, F.element_at(ordenada, meio + 1).cast("double"))
             .otherwise((F.element_at(ordenada, meio) + F.element_at(ordenada, meio + 1)) / 2.0))


def indice_mes(coluna):
    """Month count since year zero, so that month distance is a plain subtraction."""
    return (F.floor(F.col(coluna) / 100) * 12 + F.col(coluna) % 100).cast("int")

## 3. Leitura da Bronze e recorte temporal

A Bronze guarda 2023 a 2026 como a ANEEL publicou. A Silver começa em janeiro de 2024, primeiro mês em que `CodTipoManifestacao` usa diretamente o código da REH, e termina em junho de 2026, último mês consolidado. O recorte é feito pela junção com `dim_tempo`, para que o período fique definido num único lugar.

In [ ]:
bronze = spark.table(TABELA_BRONZE)
tempo = spark.table(f"{SILVER}.dim_tempo").select("ano_mes")

# Published columns only: lineage columns differ between loads and must not decide
# whether two rows are copies of each other
COLUNAS_PUBLICADAS = [c for c in bronze.columns if c not in INGESTION_COLUMNS]

com_periodo = bronze.withColumn(
    "ano_mes",
    F.col("AnoCompetencia").cast("int") * 100 + F.col("MesCompetencia").cast("int"))

recorte = com_periodo.join(F.broadcast(tempo), "ano_mes", "inner")

linhas_bronze = bronze.count()
linhas_recorte = recorte.count()

print(f"linhas na Bronze..............: {linhas_bronze:,}")
print(f"linhas de jan/2024 a jun/2026.: {linhas_recorte:,}")
print(f"fora do periodo...............: {linhas_bronze - linhas_recorte:,}")

display(com_periodo
        .join(F.broadcast(tempo), "ano_mes", "left_anti")
        .groupBy("AnoCompetencia")
        .agg(F.count("*").alias("linhas"),
             F.min("ano_mes").alias("primeiro_mes"),
             F.max("ano_mes").alias("ultimo_mes"))
        .orderBy("AnoCompetencia"))

## 4. Cópias exatas

A base publica, para o mesmo município, tipologia, canal, forma de contato e mês, mais de uma linha com quantidades diferentes. Essas linhas são parcelas de um detalhamento que a ANEEL não publica e precisam ser somadas. Entre elas há linhas idênticas em todos os 22 campos publicados, inclusive nos prazos médios de solução, que são tratadas como cópia e removidas, pela mesma regra adotada na continuidade e no INDGER.

A regra tem uma limitação que fica declarada: numa base com detalhamento oculto, duas parcelas pequenas podem coincidir em todos os campos sem serem cópia. O efeito é imaterial para o ranking, e a concentração de cópias num único mês da ELEKTRO (maio de 2026) é o padrão típico de carga duplicada. A tabela abaixo mostra onde as cópias estão.

In [ ]:
sem_copias = recorte.dropDuplicates(COLUNAS_PUBLICADAS)
linhas_sem_copias = sem_copias.count()

print(f"linhas no periodo.......: {linhas_recorte:,}")
print(f"linhas apos remocao.....: {linhas_sem_copias:,}")
print(f"copias removidas........: {linhas_recorte - linhas_sem_copias:,}")

copias = (recorte
    .groupBy(*COLUNAS_PUBLICADAS)
    .agg(F.count("*").alias("vezes"))
    .filter(F.col("vezes") > 1)
    .withColumn("extras", F.col("vezes") - 1))

display(copias
        .groupBy("SigAgente", "AnoCompetencia", "MesCompetencia")
        .agg(F.sum("extras").alias("linhas_removidas"),
             F.sum(F.col("extras") * F.col("QtdManifestacoesRecebidas").cast("long")).alias("recebidas_removidas"),
             F.sum(F.col("extras") * F.col("QtdManifestacoesProcedentes").cast("long")).alias("procedentes_removidas"))
        .orderBy(F.col("linhas_removidas").desc())
        .limit(20))

## 5. Tipagem e conformidade com a REH

Os identificadores recebem o formato das dimensões: CNPJ com 14 caracteres e zeros à esquerda, como em `dim_distribuidora`, e código de tipologia como texto, como em `dim_tipologia`. O canal publicado (`Nível 1`, `Nível 2`) vira o número do nível. `DscManifestacao` não passa para a Silver: a descrição oficial está na `dim_tipologia`.

A junção com `dim_tipologia` é interna. Código que não existe na REH não tem fonte normativa para ser classificado e fica fora do fato; a tabela abaixo mostra o que foi excluído, por código e ano.

In [ ]:
canal = F.trim(F.col("NomCanalManifestacao"))
nivel = F.lit(None).cast("int")
for rotulo, numero in NIVEL_POR_CANAL.items():
    nivel = F.when(canal == rotulo, F.lit(numero)).otherwise(nivel)

tipado = (sem_copias
    .withColumn("num_cnpj", F.lpad(F.trim(F.col("NumCPFCNPJ")), 14, "0"))
    .withColumn("cod_tipologia", F.trim(F.col("CodTipoManifestacao")))
    .withColumn("nivel", nivel)
    .withColumn("qtd_recebidas", F.col("QtdManifestacoesRecebidas").cast("long"))
    .withColumn("qtd_procedentes", F.col("QtdManifestacoesProcedentes").cast("long"))
    .withColumn("qtd_improcedentes", F.col("QtdManifestacoesImprocedentes").cast("long"))
    .select("num_cnpj", "cod_tipologia", "nivel", "ano_mes",
            "qtd_recebidas", "qtd_procedentes", "qtd_improcedentes"))

tipologia = spark.table(f"{SILVER}.dim_tipologia")

fora_reh = tipado.join(F.broadcast(tipologia.select("cod_tipologia")), "cod_tipologia", "left_anti")

resumo_fora_reh = (fora_reh
    .groupBy("cod_tipologia", F.floor(F.col("ano_mes") / 100).cast("int").alias("ano"))
    .agg(F.count("*").alias("linhas"),
         F.sum("qtd_recebidas").alias("recebidas"),
         F.sum("qtd_procedentes").alias("procedentes"))
    .orderBy("cod_tipologia", "ano"))

totais_fora = fora_reh.agg(F.count("*").alias("linhas"),
                           F.sum("qtd_recebidas").alias("recebidas"),
                           F.sum("qtd_procedentes").alias("procedentes")).first()

print(f"codigos fora da REH.....: {fora_reh.select('cod_tipologia').distinct().count()}")
print(f"linhas..................: {totais_fora['linhas']:,}")
print(f"recebidas...............: {totais_fora['recebidas'] or 0:,}")
print(f"procedentes.............: {totais_fora['procedentes'] or 0:,}")

display(resumo_fora_reh)

Dentro da REH, entram no fato apenas as reclamações detalhadas: família 102, sem linha de subtotal. Na `dim_tipologia` isso corresponde aos três blocos de reclamação. Subtotais somariam em dobro com as linhas detalhadas, e as demais famílias (informação, solicitação, denúncia, elogio e outras) não respondem a nenhuma das perguntas. A tabela mostra o volume de cada classe antes do filtro.

In [ ]:
classificado = (tipado
    .join(F.broadcast(tipologia.select("cod_tipologia", "ind_subtotal", "bloco")),
          "cod_tipologia", "inner")
    .withColumn("classe", F.when(F.col("ind_subtotal"), F.lit("subtotal"))
                           .otherwise(F.col("bloco"))))

display(classificado
        .groupBy("classe")
        .agg(F.count("*").alias("linhas"),
             F.sum("qtd_recebidas").alias("recebidas"),
             F.sum("qtd_procedentes").alias("procedentes"))
        .orderBy("classe"))

reclamacoes = classificado.filter(F.col("bloco").isin(BLOCOS_RECLAMACAO))

## 6. Agregação ao grão do fato

O município sai do grão: nenhuma pergunta de negócio o utiliza, e 22 códigos de município da base não existem na tabela do IBGE. A soma sobre município, canal de contato e parcelas não publicadas produz uma linha por CNPJ, tipologia, nível e mês. O bloco segue junto até o fim da imputação e não é gravado no fato, porque já é atributo da `dim_tipologia`.

In [ ]:
publicado = (reclamacoes
    .groupBy(*CHAVE_FATO, "bloco")
    .agg(*[F.sum(f"qtd_{m}").alias(f"qtd_{m}_publicada") for m in MEDIDAS]))

print(f"linhas no grao do fato: {publicado.count():,}")

## 7. Detecção de meses anômalos

O teste é feito por distribuidora, nível, bloco e mês, sobre o total do bloco, para cada uma das três medidas. Um mês é anômalo quando qualquer medida cai em um dos dois testes:

| Teste | Regra | O que captura |
|---|---|---|
| Mediana local | Fora de 40% a 250% da mediana dos 3 meses anteriores e 3 seguintes | Salto ou queda isolados |
| Mediana da série | Abaixo de 20% da mediana dos 30 meses da própria série | Sequência de meses ruins, que contamina os vizinhos e escapa do teste local |

O teste local só é aplicado quando a mediana dos vizinhos é positiva; com mediana zero, qualquer volume seria marcado. Ele também não marca o salto persistente: quando a mediana dos até 3 meses seguintes fica entre 40% e 250% do valor do mês, o volume mudou de patamar e o mês é o primeiro do novo nível, não uma anomalia. A regra não se aplica ao teste da série, que existe justamente para capturar sequências, nem ao último mês da série, que não tem meses seguintes para confirmar a persistência. A grade é completa: mês sem nenhuma linha publicada entra com zero, para que a ausência de envio seja detectada como anomalia em vez de passar despercebida.

### Por que só o comercial estrito é imputado

A detecção roda nos três blocos e nos dois níveis, mas a imputação fica restrita ao comercial estrito do nível 1, que é o bloco da conclusão da pergunta 1. Nos outros dois blocos o teste marca sobretudo variação real, e imputá-la apagaria justamente o que se quer medir:

| Bloco | O que o teste marca | Exemplos |
|---|---|---|
| `tecnico` | Picos de eventos climáticos reais, além das falhas de envio | CEEE-D em 2024 (enchentes no Rio Grande do Sul), ELETROPAULO em outubro de 2024 e dezembro de 2025 (temporais em São Paulo) |
| `rede_e_qualidade_outros` | Volumes de poucas unidades por mês e mudança de patamar de classificação | EQUATORIAL PI com 0 a 9 recebidas por mês; CPFL-PIRATINING passando de cerca de 350 para cerca de 3.500 recebidas a partir de abril de 2025 |

Nesses blocos o resultado do teste fica na tabela de controle como sinalizador, e as análises descritivas o consultam antes de interpretar um mês.

In [ ]:
grande_porte = (spark.table(f"{SILVER}.dim_distribuidora")
    .filter(F.col("grande_porte"))
    .select("num_cnpj"))

niveis = spark.createDataFrame([(n,) for n in sorted(NIVEL_POR_CANAL.values())], "nivel int")
blocos = spark.createDataFrame([(b,) for b in BLOCOS_RECLAMACAO], "bloco string")

grade = grande_porte.crossJoin(niveis).crossJoin(blocos).crossJoin(tempo)

total_bloco = (publicado
    .groupBy(*CHAVE_BLOCO)
    .agg(*[F.sum(f"qtd_{m}_publicada").alias(f"qtd_{m}") for m in MEDIDAS]))

serie = (grade
    .join(total_bloco, CHAVE_BLOCO, "left")
    .na.fill(0, subset=[f"qtd_{m}" for m in MEDIDAS]))

w_ordem = Window.partitionBy("num_cnpj", "nivel", "bloco").orderBy("ano_mes")
w_antes = w_ordem.rowsBetween(-MESES_VIZINHOS, -1)
w_depois = w_ordem.rowsBetween(1, MESES_VIZINHOS)
w_serie = Window.partitionBy("num_cnpj", "nivel", "bloco")

sinais = []
for m in MEDIDAS:
    valor = F.col(f"qtd_{m}")
    vizinhos = F.concat(F.collect_list(valor).over(w_antes), F.collect_list(valor).over(w_depois))
    serie = (serie
        .withColumn(f"med_local_{m}", mediana(vizinhos))
        .withColumn(f"med_serie_{m}", mediana(F.collect_list(valor).over(w_serie))))

    # A jump that the following months confirm is a new level, not an anomaly
    serie = serie.withColumn(f"med_seguinte_{m}", mediana(F.collect_list(valor).over(w_depois)))
    seguinte = F.col(f"med_seguinte_{m}")
    persistente = F.coalesce(seguinte.isNotNull() &
                             (seguinte >= LIMITE_INFERIOR_LOCAL * valor) &
                             (seguinte <= LIMITE_SUPERIOR_LOCAL * valor), F.lit(False))

    local = F.col(f"med_local_{m}")
    fora_local = ((local > 0) & ~persistente &
                  ((valor < LIMITE_INFERIOR_LOCAL * local) |
                   (valor > LIMITE_SUPERIOR_LOCAL * local)))
    fora_serie = valor < LIMITE_INFERIOR_SERIE * F.col(f"med_serie_{m}")

    serie = (serie
        .withColumn(f"anomalo_local_{m}", F.coalesce(fora_local, F.lit(False)))
        .withColumn(f"anomalo_serie_{m}", F.coalesce(fora_serie, F.lit(False))))
    sinais += [F.col(f"anomalo_local_{m}"), F.col(f"anomalo_serie_{m}")]

mes_anomalo = sinais[0]
for s in sinais[1:]:
    mes_anomalo = mes_anomalo | s

serie = serie.withColumn("mes_anomalo", mes_anomalo)

### Meses a imputar e vizinhos válidos

Um mês é imputado quando é anômalo, pertence ao nível 1 e a um bloco imputável, não é o último mês da série e tem ao menos um mês vizinho válido. Vizinho válido é mês a até 3 meses de distância, dentro da série e não anômalo no mesmo bloco. No primeiro mês da série só existem os vizinhos seguintes.

O último mês da série não é imputado. Sem meses seguintes, não há como distinguir anomalia de mudança de patamar, que é justamente o que a regra do salto persistente verifica; imputar nesse caso seria decidir sem evidência. O mês fica marcado como anômalo na tabela de controle, com o valor publicado.

In [ ]:
indice = tempo.withColumn("idx", indice_mes("ano_mes"))

# The last month has no following months to tell an anomaly from a new level
ULTIMO_MES = tempo.agg(F.max("ano_mes")).first()[0]

anomalos_imputaveis = (serie
    .filter((F.col("nivel") == NIVEL_IMPUTADO) &
            F.col("bloco").isin(BLOCOS_IMPUTADOS) &
            F.col("mes_anomalo"))
    .select(*CHAVE_BLOCO))

a_imputar = anomalos_imputaveis.filter(F.col("ano_mes") < ULTIMO_MES)

vizinhos = (a_imputar
    .withColumn("idx", indice_mes("ano_mes"))
    .join(indice.select(F.col("ano_mes").alias("ano_mes_vizinho"), F.col("idx").alias("idx_vizinho")),
          (F.abs(F.col("idx_vizinho") - F.col("idx")) >= 1) &
          (F.abs(F.col("idx_vizinho") - F.col("idx")) <= MESES_VIZINHOS))
    .join(anomalos_imputaveis.select("num_cnpj", "nivel", "bloco", F.col("ano_mes").alias("ano_mes_vizinho")),
          ["num_cnpj", "nivel", "bloco", "ano_mes_vizinho"], "left_anti")
    .select(*CHAVE_BLOCO, "ano_mes_vizinho"))

qtd_vizinhos = vizinhos.groupBy(*CHAVE_BLOCO).agg(F.count("*").alias("meses_vizinhos_validos"))

## 8. `controle_anomalias_manifestacao`

Uma linha por distribuidora de grande porte, nível, bloco e mês, com o total publicado do bloco, as medianas de referência, o resultado de cada teste e a marcação de imputação. É a evidência que sustenta a imputação e o sinalizador dos demais blocos e do nível 2, e a fonte da lista de casos citados na autoavaliação. A tabela é gravada antes do fato porque a imputação lê dela a lista de meses a imputar.

In [ ]:
controle = (serie
    .join(a_imputar.join(qtd_vizinhos, CHAVE_BLOCO, "inner"), CHAVE_BLOCO, "left")
    .withColumn("imputado", F.col("meses_vizinhos_validos").isNotNull())
    .join(a_imputar.withColumn("imputavel", F.lit(True)), CHAVE_BLOCO, "left")
    .withColumn("imputavel", F.coalesce(F.col("imputavel"), F.lit(False))))

(controle.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.controle_anomalias_manifestacao"))

controle_lido = spark.table(f"{SILVER}.controle_anomalias_manifestacao")
nomes = spark.table(f"{SILVER}.dim_distribuidora").select("num_cnpj", "sig_agente")

print(f"controle_anomalias_manifestacao: {controle_lido.count():,} linhas")
print(f"meses anomalos por nivel e bloco:")
display(controle_lido
        .groupBy("nivel", "bloco")
        .agg(F.sum(F.col("mes_anomalo").cast("int")).alias("meses_anomalos"),
             F.sum(F.col("imputado").cast("int")).alias("meses_imputados"))
        .orderBy("nivel", "bloco"))

display(controle_lido
        .filter("imputavel")
        .join(nomes, "num_cnpj")
        .select("sig_agente", "ano_mes",
                "qtd_recebidas", "med_local_recebidas",
                "qtd_procedentes", "med_local_procedentes",
                "qtd_improcedentes", "med_local_improcedentes",
                "meses_vizinhos_validos", "imputado")
        .orderBy("sig_agente", "ano_mes"))

In [ ]:
COMENTARIOS_CONTROLE = {
    "num_cnpj": "CNPJ da distribuidora de grande porte",
    "nivel": "Nivel de atendimento: 1 (central de atendimento) ou 2 (ouvidoria)",
    "bloco": "Bloco da dim_tipologia: comercial_estrito, rede_e_qualidade_outros ou tecnico",
    "ano_mes": "Ano e mes de competencia no formato AAAAMM",
    "mes_anomalo": "Verdadeiro quando alguma medida falha no teste local ou no teste da serie",
    "imputavel": "Verdadeiro quando o mes e anomalo, do nivel 1, de bloco sujeito a imputacao e anterior ao ultimo mes da serie",
    "meses_vizinhos_validos": "Meses vizinhos nao anomalos usados na imputacao; nulo quando o mes nao foi imputado",
    "imputado": "Verdadeiro quando o mes e imputavel e teve ao menos um vizinho valido",
}
for m in MEDIDAS:
    COMENTARIOS_CONTROLE.update({
        f"qtd_{m}": f"Total publicado de {m} no bloco e mes; zero quando nada foi publicado",
        f"med_local_{m}": f"Mediana de {m} nos 3 meses anteriores e 3 seguintes do mesmo bloco",
        f"med_serie_{m}": f"Mediana de {m} nos 30 meses da serie do mesmo bloco",
        f"med_seguinte_{m}": f"Mediana de {m} nos ate 3 meses seguintes; confirma salto persistente",
        f"anomalo_local_{m}": f"Verdadeiro quando {m} fica fora de 40 a 250 por cento da mediana local sem persistir nos meses seguintes",
        f"anomalo_serie_{m}": f"Verdadeiro quando {m} fica abaixo de 20 por cento da mediana da serie",
    })

spark.sql(f"""COMMENT ON TABLE {SILVER}.controle_anomalias_manifestacao IS
    'Teste de meses anomalos das reclamacoes por distribuidora de grande porte, nivel, bloco
     e mes. Evidencia da imputacao do comercial estrito no nivel 1 e sinalizador dos demais
     blocos e do nivel 2.'""")

for coluna, texto in COMENTARIOS_CONTROLE.items():
    spark.sql(f"ALTER TABLE {SILVER}.controle_anomalias_manifestacao "
              f"ALTER COLUMN {coluna} COMMENT '{texto}'")

print(f"comentarios aplicados: {len(COMENTARIOS_CONTROLE)} colunas")

## 9. Imputação

Para cada linha (distribuidora × tipologia) de um mês imputado, as três medidas recebem a mediana da mesma linha nos meses vizinhos válidos. Linha ausente num mês vizinho conta como zero, e a tipologia que aparece nos vizinhos mas não no mês anômalo também é imputada, o que permite recompor um mês em que parte das tipologias deixou de ser enviada.

A mediana de um número par de meses é a média dos dois centrais e pode não ser inteira; o valor é arredondado, por se tratar de contagem. Como as três medidas são imputadas de forma independente, procedentes mais improcedentes não precisam somar exatamente as recebidas no mês imputado, o que já ocorre no dado publicado, em que parte das manifestações ainda não tem conclusão.

In [ ]:
imputados = controle_lido.filter("imputado").select(*CHAVE_BLOCO)
vizinhos_lidos = vizinhos.join(imputados, CHAVE_BLOCO, "inner")

# Rows to impute: typologies published in the anomalous month or in any valid neighbour
linhas_mes = (publicado.join(imputados, CHAVE_BLOCO, "inner")
    .select(*CHAVE_BLOCO, "cod_tipologia"))
linhas_vizinhas = (publicado
    .withColumnRenamed("ano_mes", "ano_mes_vizinho")
    .join(vizinhos_lidos, ["num_cnpj", "nivel", "bloco", "ano_mes_vizinho"], "inner")
    .select(*CHAVE_BLOCO, "cod_tipologia"))
candidatas = linhas_mes.unionByName(linhas_vizinhas).distinct()

valores_vizinhos = (candidatas
    .join(vizinhos_lidos, CHAVE_BLOCO, "inner")
    .join(publicado.withColumnRenamed("ano_mes", "ano_mes_vizinho"),
          ["num_cnpj", "nivel", "bloco", "cod_tipologia", "ano_mes_vizinho"], "left")
    .na.fill(0, subset=[f"qtd_{m}_publicada" for m in MEDIDAS]))

imputacao = (valores_vizinhos
    .groupBy(*CHAVE_FATO, "bloco")
    .agg(*[F.collect_list(f"qtd_{m}_publicada").alias(f"lista_{m}") for m in MEDIDAS])
    .select(*CHAVE_FATO, "bloco",
            *[F.round(mediana(F.col(f"lista_{m}")), 0).cast("long").alias(f"qtd_{m}_imputada")
              for m in MEDIDAS]))

## 10. `fato_manifestacao`

As colunas `qtd_*` são as que a Gold usa: valor imputado no mês imputado, valor publicado nos demais. As colunas `qtd_*_publicada` guardam o que a ANEEL publicou, com zero para a tipologia que não constava do mês e foi recomposta pela imputação. O ranking sem imputação, que a Gold publica como sensibilidade, é calculado sobre essas colunas.

In [ ]:
base = publicado.withColumn("publicada", F.lit(True))

fato = (base
    .join(imputacao, CHAVE_FATO + ["bloco"], "full_outer")
    .join(imputados.withColumn("imputado", F.lit(True)), CHAVE_BLOCO, "left")
    .withColumn("imputado", F.coalesce(F.col("imputado"), F.lit(False)))
    .withColumn("publicada", F.coalesce(F.col("publicada"), F.lit(False)))
    .na.fill(0, subset=[f"qtd_{m}_publicada" for m in MEDIDAS]))

for m in MEDIDAS:
    fato = fato.withColumn(
        f"qtd_{m}",
        F.when(F.col("imputado"), F.coalesce(F.col(f"qtd_{m}_imputada"), F.lit(0)))
         .otherwise(F.col(f"qtd_{m}_publicada")))

# A row created only by the imputation and imputed as zero carries no information
sem_informacao = ~F.col("publicada") & (sum(F.col(f"qtd_{m}") for m in MEDIDAS) == 0)

COLUNAS_FATO = (CHAVE_FATO
                + [f"qtd_{m}" for m in MEDIDAS]
                + [f"qtd_{m}_publicada" for m in MEDIDAS]
                + ["imputado"])

fato_final = fato.filter(~sem_informacao).select(*COLUNAS_FATO)

(fato_final.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.fato_manifestacao"))

fato_lido = spark.table(f"{SILVER}.fato_manifestacao")
print(f"fato_manifestacao: {fato_lido.count():,} linhas, {len(fato_lido.columns)} colunas")
print(f"linhas imputadas.: {fato_lido.filter('imputado').count():,}")

display(fato_lido
        .filter("imputado")
        .join(nomes, "num_cnpj")
        .groupBy("sig_agente", "ano_mes")
        .agg(*[F.sum(f"qtd_{m}_publicada").alias(f"{m}_publicada") for m in MEDIDAS],
             *[F.sum(f"qtd_{m}").alias(f"{m}_usada") for m in MEDIDAS])
        .orderBy("sig_agente", "ano_mes"))

In [ ]:
COMENTARIOS_FATO = {
    "num_cnpj": "CNPJ da distribuidora, texto de 14 caracteres com zeros a esquerda",
    "cod_tipologia": "Codigo da tipologia conforme Anexo I da REH 2.992/2021; apenas reclamacoes detalhadas da familia 102",
    "nivel": "Nivel de atendimento: 1 (central de atendimento) ou 2 (ouvidoria da distribuidora)",
    "ano_mes": "Ano e mes de competencia no formato AAAAMM",
    "qtd_recebidas": "Manifestacoes recebidas usadas no calculo: imputadas no mes imputado, publicadas nos demais",
    "qtd_procedentes": "Reclamacoes procedentes usadas no calculo: imputadas no mes imputado, publicadas nos demais",
    "qtd_improcedentes": "Reclamacoes improcedentes usadas no calculo: imputadas no mes imputado, publicadas nos demais",
    "qtd_recebidas_publicada": "Manifestacoes recebidas conforme publicadas pela ANEEL, somadas no grao do fato",
    "qtd_procedentes_publicada": "Reclamacoes procedentes conforme publicadas pela ANEEL, somadas no grao do fato",
    "qtd_improcedentes_publicada": "Reclamacoes improcedentes conforme publicadas pela ANEEL, somadas no grao do fato",
    "imputado": "Verdadeiro quando a linha pertence a mes anomalo imputado (comercial estrito, nivel 1)",
}

spark.sql(f"""COMMENT ON TABLE {SILVER}.fato_manifestacao IS
    'Reclamacoes detalhadas da familia 102 da REH 2.992/2021 no grao CNPJ, tipologia, nivel
     e mes, de janeiro de 2024 a junho de 2026. Guarda o valor usado e o publicado lado a
     lado; o universo de distribuidoras e aplicado na Gold.'""")

for coluna, texto in COMENTARIOS_FATO.items():
    spark.sql(f"ALTER TABLE {SILVER}.fato_manifestacao "
              f"ALTER COLUMN {coluna} COMMENT '{texto}'")

print(f"comentarios aplicados: {len(COMENTARIOS_FATO)} colunas")

## 11. Achados de qualidade

A execução das seções anteriores revelou dois casos que as regras gerais não resolviam. Os dois foram identificados ao confrontar o valor imputado com o comportamento da série, e não pelo teste de anomalia em si.

### 11.1 CELESC: série internamente inconsistente

No comercial estrito do nível 1, a CELESC é a única das distribuidoras de grande porte com meses em que as procedentes superam as recebidas. Entre 2024 e 2025 as recebidas oscilam de cerca de 5 mil para cerca de 26 mil por mês, e as procedentes chegam a mais de 36 mil. O teste marcou três meses, mas a imputação piorou o dado: em junho de 2025 as procedentes passaram de 7.989 para 23.181, porque os meses vizinhos, usados como referência, também estão inflados.

O problema é a série, não meses isolados, e nenhum tratamento na Silver a torna confiável. A decisão é excluir a CELESC do ranking de reclamações. A exclusão é registrada em `exclusao_ranking_manifestacao`, com o motivo e a evidência calculada do próprio dado, e aplicada na Gold; a Silver mantém as linhas da distribuidora, que continuam disponíveis para consulta. A causa não foi apurada.

In [ ]:
consistencia = (controle_lido
    .filter((F.col("nivel") == NIVEL_IMPUTADO) & (F.col("bloco") == "comercial_estrito"))
    .groupBy("num_cnpj")
    .agg(F.sum((F.col("qtd_procedentes") > F.col("qtd_recebidas")).cast("int"))
          .alias("meses_procedentes_acima_recebidas")))

print("distribuidoras com procedentes acima de recebidas no comercial estrito do nivel 1:")
display(consistencia
        .filter(F.col("meses_procedentes_acima_recebidas") > 0)
        .join(nomes, "num_cnpj")
        .select("sig_agente", "num_cnpj", "meses_procedentes_acima_recebidas"))

display(controle_lido
        .filter((F.col("nivel") == NIVEL_IMPUTADO) & (F.col("bloco") == "comercial_estrito") &
                F.col("num_cnpj").isin(list(EXCLUSOES_RANKING)))
        .join(nomes, "num_cnpj")
        .select("sig_agente", "ano_mes", "qtd_recebidas", "qtd_procedentes",
                "qtd_improcedentes", "mes_anomalo", "imputado")
        .orderBy("sig_agente", "ano_mes"))

In [ ]:
exclusao = (spark.createDataFrame(list(EXCLUSOES_RANKING.items()), "num_cnpj string, motivo string")
    .join(consistencia, "num_cnpj", "left")
    .withColumn("meses_procedentes_acima_recebidas",
                F.coalesce(F.col("meses_procedentes_acima_recebidas"), F.lit(0)).cast("int")))

(exclusao.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER}.exclusao_ranking_manifestacao"))

COMENTARIOS_EXCLUSAO = {
    "num_cnpj": "CNPJ da distribuidora excluida do ranking de reclamacoes",
    "motivo": "Motivo da exclusao, registrado na decisao metodologica",
    "meses_procedentes_acima_recebidas": "Evidencia: meses do comercial estrito do nivel 1 com procedentes acima de recebidas",
}

spark.sql(f"""COMMENT ON TABLE {SILVER}.exclusao_ranking_manifestacao IS
    'Distribuidoras excluidas do ranking de reclamacoes por qualidade insuficiente do dado,
     com motivo e evidencia. A Silver mantem as linhas; a Gold aplica a exclusao.'""")

for coluna, texto in COMENTARIOS_EXCLUSAO.items():
    spark.sql(f"ALTER TABLE {SILVER}.exclusao_ranking_manifestacao "
              f"ALTER COLUMN {coluna} COMMENT '{texto}'")

display(spark.table(f"{SILVER}.exclusao_ranking_manifestacao").join(nomes, "num_cnpj", "left"))

### 11.2 CEEE-D: mudança de patamar

Na primeira execução, a CEEE-D foi marcada em abril de 2026 pelo teste local: as recebidas do comercial estrito subiram de cerca de 3.900 para 9.899. A imputação reduziria abril para 4.040, mas maio e junho mantêm o novo nível, com 9.265 e 9.402, e o resultado seria uma queda artificial restrita a abril. O caso levou à regra do salto persistente descrita na seção 7. A tabela abaixo mostra o mês com a mediana dos meses seguintes que confirma o novo patamar.

In [ ]:
CASO_PATAMAR = ("08467115000100", 202604)   # CEEE-D, April 2026

display(controle_lido
        .filter((F.col("num_cnpj") == CASO_PATAMAR[0]) & (F.col("nivel") == NIVEL_IMPUTADO) &
                (F.col("bloco") == "comercial_estrito") &
                F.col("ano_mes").between(CASO_PATAMAR[1] - 3, CASO_PATAMAR[1] + 2))
        .select("ano_mes", "qtd_recebidas", "med_local_recebidas", "med_seguinte_recebidas",
                "anomalo_local_recebidas", "mes_anomalo", "imputado")
        .orderBy("ano_mes"))

## 12. Validação

Se sujeira chegou à Gold, o erro está aqui. Os testes cobrem a chave, a conformidade com as três dimensões, a conservação do volume entre Bronze e fato, a coerência entre valor usado e publicado e a detecção dos casos conhecidos da fase exploratória: CELESC em maio de 2024, junho de 2025 e dezembro de 2025 e Neoenergia Brasília em junho de 2026, no comercial estrito, e ELEKTRO em junho de 2026, no bloco técnico.

In [ ]:
testes = []

linhas = fato_lido.count()
unicas = fato_lido.select(*CHAVE_FATO).distinct().count()
testes.append(("chave unica no grao do fato",
               linhas == unicas,
               f"{linhas:,} linhas para {unicas:,} chaves"))

sem_nivel = tipado.filter(F.col("nivel").isNull()).count()
testes.append(("todo canal mapeado para nivel",
               sem_nivel == 0,
               f"{sem_nivel} linhas com canal nao reconhecido"))

sem_distribuidora = (fato_lido.select("num_cnpj").distinct()
    .join(spark.table(f"{SILVER}.dim_distribuidora"), "num_cnpj", "left_anti").count())
testes.append(("todo CNPJ na dim_distribuidora",
               sem_distribuidora == 0,
               f"{sem_distribuidora} CNPJs sem correspondencia"))

fora_tempo = fato_lido.join(tempo, "ano_mes", "left_anti").count()
testes.append(("todo mes na dim_tempo",
               fora_tempo == 0,
               f"{fora_tempo} linhas fora de jan/2024 a jun/2026"))

fora_recorte = (fato_lido.join(tipologia, "cod_tipologia", "inner")
    .filter(~F.col("bloco").isin(BLOCOS_RECLAMACAO) | F.col("ind_subtotal")).count())
testes.append(("apenas reclamacoes detalhadas",
               fora_recorte == 0,
               f"{fora_recorte} linhas de subtotal ou fora da familia 102"))

procedentes_total = tipado.agg(F.sum("qtd_procedentes")).first()[0]
procedentes_fora = totais_fora["procedentes"] or 0
testes.append(("volume fora da REH imaterial",
               procedentes_fora <= 0.0001 * procedentes_total,
               f"{procedentes_fora:,} procedentes de {procedentes_total:,}"))

esperado = reclamacoes.agg(*[F.sum(f"qtd_{m}").alias(m) for m in MEDIDAS]).first()
obtido = fato_lido.agg(*[F.sum(f"qtd_{m}_publicada").alias(m) for m in MEDIDAS]).first()
testes.append(("publicado conservado entre Bronze e fato",
               all(esperado[m] == obtido[m] for m in MEDIDAS),
               " | ".join(f"{m} {obtido[m]:,}/{esperado[m]:,}" for m in MEDIDAS)))

divergentes = fato_lido.filter(~F.col("imputado") & (
    (F.col("qtd_recebidas") != F.col("qtd_recebidas_publicada")) |
    (F.col("qtd_procedentes") != F.col("qtd_procedentes_publicada")) |
    (F.col("qtd_improcedentes") != F.col("qtd_improcedentes_publicada")))).count()
testes.append(("usado igual ao publicado fora da imputacao",
               divergentes == 0,
               f"{divergentes} linhas divergentes"))

negativos = fato_lido.filter(
    (F.col("qtd_recebidas") < 0) | (F.col("qtd_procedentes") < 0) |
    (F.col("qtd_improcedentes") < 0)).count()
testes.append(("quantidades nao negativas",
               negativos == 0,
               f"{negativos} linhas com valor negativo"))

imputado_fora = (fato_lido.filter("imputado")
    .join(tipologia.select("cod_tipologia", "bloco"), "cod_tipologia")
    .filter((F.col("nivel") != NIVEL_IMPUTADO) | ~F.col("bloco").isin(BLOCOS_IMPUTADOS))
    .count())
testes.append(("imputacao restrita ao escopo definido",
               imputado_fora == 0,
               f"{imputado_fora} linhas imputadas fora de nivel {NIVEL_IMPUTADO} e {BLOCOS_IMPUTADOS}"))

sem_vizinho = controle_lido.filter(F.col("imputavel") & ~F.col("imputado")).count()
testes.append(("todo mes imputavel imputado",
               sem_vizinho == 0,
               f"{sem_vizinho} meses sem vizinho valido"))

meses_serie = tempo.count()
incompletas = (controle_lido.groupBy("num_cnpj", "nivel", "bloco").count()
    .filter(F.col("count") != meses_serie).count())
testes.append(("grade completa no controle",
               incompletas == 0,
               f"{incompletas} series sem os {meses_serie} meses"))

CASOS_CONHECIDOS = [
    ("08336783000190", "comercial_estrito", 202405),   # CELESC
    ("08336783000190", "comercial_estrito", 202506),   # CELESC
    ("08336783000190", "comercial_estrito", 202512),   # CELESC
    ("07522669000192", "comercial_estrito", 202606),   # Neoenergia Brasilia
    ("02328280000197", "tecnico", 202606),             # ELEKTRO
]
marcados = {(r["num_cnpj"], r["bloco"], r["ano_mes"]) for r in
            controle_lido.filter((F.col("nivel") == NIVEL_IMPUTADO) & F.col("mes_anomalo"))
                         .select("num_cnpj", "bloco", "ano_mes").collect()}
nao_detectados = [c for c in CASOS_CONHECIDOS if c not in marcados]
testes.append(("casos conhecidos detectados",
               not nao_detectados,
               f"{len(CASOS_CONHECIDOS) - len(nao_detectados)} de {len(CASOS_CONHECIDOS)}"
               + (f"; faltam {nao_detectados}" if nao_detectados else "")))

exclusao_lida = spark.table(f"{SILVER}.exclusao_ranking_manifestacao")
sem_evidencia = exclusao_lida.filter(F.col("meses_procedentes_acima_recebidas") <= 0).count()
testes.append(("exclusao sustentada pelo dado",
               sem_evidencia == 0 and exclusao_lida.count() == len(EXCLUSOES_RANKING),
               f"{sem_evidencia} exclusoes sem evidencia"))

inconsistentes = (consistencia
    .filter(F.col("meses_procedentes_acima_recebidas") > 0)
    .filter(~F.col("num_cnpj").isin(list(EXCLUSOES_RANKING))).count())
testes.append(("inconsistencia restrita as excluidas",
               inconsistentes == 0,
               f"{inconsistentes} distribuidoras inconsistentes fora da exclusao"))

caso_patamar = (controle_lido
    .filter((F.col("num_cnpj") == CASO_PATAMAR[0]) & (F.col("ano_mes") == CASO_PATAMAR[1]) &
            (F.col("nivel") == NIVEL_IMPUTADO) & (F.col("bloco") == "comercial_estrito"))
    .select("mes_anomalo").first()["mes_anomalo"])
testes.append(("salto persistente nao marcado",
               not caso_patamar,
               f"CEEE-D {CASO_PATAMAR[1]} marcado como anomalo: {caso_patamar}"))

imputado_ultimo = fato_lido.filter(F.col("imputado") & (F.col("ano_mes") == ULTIMO_MES)).count()
testes.append(("ultimo mes da serie nao imputado",
               imputado_ultimo == 0,
               f"{imputado_ultimo} linhas imputadas em {ULTIMO_MES}"))

for nome, passou, detalhe in testes:
    print(f"[{'OK' if passou else 'FALHOU':<7}] {nome:<45} {detalhe}")

if all(p for _, p, _ in testes):
    print("\nSilver de reclamacoes validada.")
else:
    print("\nHa teste sem passar; corrigir antes de seguir para a Gold.")

## Pendências documentadas

| Item | Situação | O que falta |
|---|---|---|
| Cópias exatas numa base com detalhamento oculto | Removidas pela regra do projeto; efeito imaterial | Declarar a limitação na autoavaliação |
| Visual de contribuição dos grupos do nível 2 da REH | Planejado para o notebook 03 | Construir sobre `fato_manifestacao` e `dim_tipologia` |
| Ranking sem imputação | Colunas `qtd_*_publicada` disponíveis | Calcular na Gold como sensibilidade |
| Exclusão da CELESC | Registrada em `exclusao_ranking_manifestacao` | Aplicar na Gold; causa da inconsistência não apurada |
| Último mês da série | Não imputado, por não haver meses seguintes que distingam anomalia de novo patamar | Neoenergia Brasília em junho de 2026 fica com o valor publicado e a marcação de anomalia; reavaliar quando julho de 2026 estiver consolidado |

## Autoavaliação desta etapa

### O que a etapa entregou

A Silver de reclamações com três tabelas: `fato_manifestacao`, no grão distribuidora × tipologia × nível × mês, com o valor usado e o publicado lado a lado; `controle_anomalias_manifestacao`, com o teste de cada mês, bloco e nível das distribuidoras de grande porte; e `exclusao_ranking_manifestacao`. O volume publicado é conservado da Bronze ao fato, os códigos fora da REH e as cópias exatas ficam quantificados, e toda alteração de valor é rastreável até o mês e a regra que a produziram.

### O que mudou no caminho

- A imputação, prevista para os três blocos, ficou restrita ao comercial estrito. Nos blocos técnico e de rede, o teste marcava sobretudo variação real, como enchentes e temporais, e mudanças de patamar de classificação.
- O teste local ganhou a regra do salto persistente, depois que a imputação da CEEE-D em abril de 2026 produziu uma queda artificial.
- A CELESC foi excluída do ranking: a série é internamente inconsistente, e a imputação piorava o dado em vez de corrigi-lo.
- A contagem dos códigos fora da REH, prevista para o notebook de qualidade da Bronze, ficou aqui, junto da exclusão que ela sustenta.

### O que ficou em aberto

- A causa da inconsistência na CELESC não foi apurada.
- A remoção de cópias exatas não distingue cópia de coincidência entre parcelas de um detalhamento não publicado. O efeito é imaterial, mas a limitação existe.
- O último mês da série não é imputado, por não permitir confirmar a persistência de um salto; a Neoenergia Brasília em junho de 2026 fica com o valor publicado. Sem imputação, a distribuidora cai do 17º para o 30º lugar no ranking do comercial estrito, posição coerente com os dois semestres anteriores, em que ela já estava entre as últimas.
- Nível 2 e blocos técnico e de rede têm apenas o sinalizador, sem tratamento.

### O que eu faria diferente

Eu testaria a consistência interna da série, como procedentes não superarem recebidas, antes da detecção de anomalias: a CELESC teria saído do processo antes de passar pela imputação. Também confrontaria cada valor imputado com os meses seguintes antes de aceitá-lo, porque foi essa comparação, e não o teste de anomalia, que revelou os dois achados da seção 11. Por fim, gravaria o agregado da seção 6 como tabela intermediária, para que as etapas seguintes não relessem a Bronze inteira a cada consulta.